<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_07_model_analysis/stage_07_00_model_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_00 – Model Analysis**

In [ ]:
import pandas as pd
from pathlib import Path
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# **1. Métricas de SEQ2ONE**

In [ ]:
import pandas as pd
from pathlib import Path

def load_all_seq2one_metrics(
    *,
    models: list[str],
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    """
    Carga métricas seq2one para múltiples modelos y
    mantiene solo columnas estándar comparables.
    """

    cols = [
        "model", "split", "window_size", "target",
        "horizon_min", "MAE", "RMSE", "R2", "DA"
    ]

    dfs = []

    base_path = Path(base_dir)

    for name in models:
        path = base_path / f"seq2one_{name}_metrics.parquet"

        if not path.exists():
            continue

        df = pd.read_parquet(path)

        # Mantener solo columnas deseadas si existen
        keep_cols = [c for c in cols if c in df.columns]
        df = df[keep_cols].copy()

        dfs.append(df)

    if not dfs:
        return pd.DataFrame(columns=cols)

    df_all = pd.concat(dfs, ignore_index=True)

    # Orden consistente
    df_all = (
        df_all
        .sort_values(["model", "window_size", "target", "split"])
        .reset_index(drop=True)
    )

    return df_all


In [ ]:
models_seq2one = ['naive', 'ridge', 'lasso', 'mlp', 'gru', 'lstm', 'tcn', 'transformer']

df_seq2one_all = load_all_seq2one_metrics(
    models=models_seq2one,
    base_dir="/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics"
)

In [ ]:
df_seq2one_all

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA
0,gru,test,30,delta_60,60,48.576339,76.975064,0.115642,0.619139
1,gru,valid,30,delta_60,60,30.210935,43.738906,0.208353,0.647863
2,gru,test,30,delta_90,90,62.513185,98.251515,0.078711,0.598776
3,gru,valid,30,delta_90,90,39.720030,57.446933,0.128378,0.624933
4,gru,test,30,ret_60,60,0.005739,0.008296,-2.881864,0.499769
...,...,...,...,...,...,...,...,...,...
275,transformer,valid,180,delta_90,90,31.532164,48.172492,0.369248,0.716830
276,transformer,test,180,ret_60,60,0.005443,0.008678,-3.120200,0.530955
277,transformer,valid,180,ret_60,60,0.003852,0.004837,-2.334733,0.554008
278,transformer,test,180,ret_90,90,0.009241,0.011280,-3.576115,0.534092


## **1.1. Selección del tipo de target**

### **1.1.1. Criterios técnicos**

**Criterios primarios**

- R² > 0 de forma consistente entre modelos.
- MAE estable y razonable.
- Directional Accuracy (DA) claramente superior a 0.55.
- Baja varianza de desempeño entre modelos.

**Observación preliminar**

- `delta_60` y `delta_90` presentan R² positivos y razonables.
- `ret_60` y `ret_90` muestran R² negativos en la mayoría de los casos.

**Decisión metodológica**

Primero se debe seleccionar entre `delta` y `ret`.

Si el objetivo es obtener una señal predictiva explotable, el target elegido debe:

- Presentar R² positivo en la mayoría de los modelos.
- Tener mayor DA promedio.
- Mostrar menor dispersión de resultados entre arquitecturas.

Preliminarmente, el candidato más sólido parece ser `delta_60`.

**Criterio de decisión: uso de R² como métrica principal**

**1. Qué mide R²**

R² mide la **proporción de varianza explicada** por el modelo respecto a un baseline constante.

En regresión:

R² = 1 − (MSE_model / MSE_baseline)

Donde el baseline es predecir siempre la media del target.

Interpretación:

- R² > 0 → el modelo mejora al baseline.
- R² = 0 → el modelo es equivalente al baseline.
- R² < 0 → el modelo es peor que el baseline.

Por lo tanto, R² es una métrica estructural que indica si existe capacidad explicativa real.

---

**2. Relevancia para este problema**

En esta etapa el objetivo no es aún maximizar PnL, sino:

> Evaluar si el target es predecible.

R² responde directamente a esa pregunta.

MAE y RMSE solo miden magnitud del error absoluto, pero no indican si el modelo mejora significativamente respecto a un baseline simple.

Un modelo puede tener MAE bajo y aun así no explicar varianza relevante si el target tiene poca dispersión.

---

**3. Interpretación de R² en series financieras**

En problemas financieros:

- R² ≈ 5% ya es interesante.
- R² ≈ 20–30% es muy fuerte.
- R² negativo implica ausencia de señal explotable.

Usar R² como criterio principal equivale a preguntar:

> ¿Existe señal estructural o estamos modelando ruido?

---

**4. Por qué MAE no es la métrica principal**

Limitaciones del MAE:

- Depende de la escala del target.
- No es relativo a un baseline.
- No permite comparar fácilmente targets con distinta varianza.

Ejemplo:

- delta_60 puede tener MAE = 25
- ret_60 puede tener MAE = 0.002

No son directamente comparables.

R² sí permite comparación transversal entre targets.

---

**5. Por qué DA no puede ser el criterio central**

Directional Accuracy (DA):

- Ignora magnitud del error.
- Puede inflarse si el target tiene sesgo estructural.
- No penaliza errores grandes.

DA es útil como métrica complementaria, pero no como criterio principal de selección.

---

**6. Orden metodológico en esta etapa**

Para la selección del tipo de target:

1. R² → evaluar capacidad explicativa.
2. DA → validar señal direccional.
3. MAE → analizar estabilidad del error.
4. Varianza del desempeño → evaluar robustez entre modelos.

---

**Conclusión**

R² se utiliza como métrica principal porque:

- Es relativa al baseline.
- Mide capacidad explicativa real.
- Permite comparar targets con distinta escala.
- Responde a la pregunta fundamental: si existe señal estructural en el target.

### **1.1.2. Implementación**

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
import pandas as pd
import numpy as np

# -------------------------------------------------
# Configuración
# -------------------------------------------------
DF = df_seq2one_all.copy()
SPLIT = "valid"

# Filtrar solo VALID
df_valid = DF[DF["split"] == SPLIT].copy()

In [ ]:
import pandas as pd
import numpy as np

# Requiere df_valid con columnas:
# ['model','split','window_size','target','horizon_min','MAE','RMSE','R2','DA']

required = {"model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA"}
missing = required - set(df_valid.columns)
if missing:
    raise ValueError(f"df_valid no tiene columnas requeridas: {sorted(missing)}")

df = df_valid.copy()
df = df[df["split"].astype(str).str.lower().eq("valid")].copy()

# Normalización mínima
df["target"] = df["target"].astype(str).str.lower().str.strip()

# Tipo de target: delta vs ret
df["target_type"] = np.where(df["target"].str.startswith("delta"), "delta",
                      np.where(df["target"].str.startswith("ret"), "ret", "other"))

# Flags de criterios primarios
df["r2_pos"] = df["R2"] > 0
df["da_gt_055"] = df["DA"] > 0.55

def summarize(g: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "n_rows": len(g),
        "n_models": g["model"].nunique(),
        "n_windows": g["window_size"].nunique(),

        "R2_mean": g["R2"].mean(),
        "R2_median": g["R2"].median(),
        "R2_std": g["R2"].std(ddof=0),
        "R2_share_pos": g["r2_pos"].mean(),

        "DA_mean": g["DA"].mean(),
        "DA_median": g["DA"].median(),
        "DA_std": g["DA"].std(ddof=0),
        "DA_share_gt_055": g["da_gt_055"].mean(),

        "MAE_mean": g["MAE"].mean(),
        "MAE_median": g["MAE"].median(),
        "MAE_std": g["MAE"].std(ddof=0),
    })

def fmt_table(t: pd.DataFrame) -> pd.DataFrame:
    out = t.copy()
    # porcentajes
    for c in ["R2_share_pos", "DA_share_gt_055"]:
        if c in out.columns:
            out[c] = (out[c] * 100).round(1).astype(str) + "%"
    # redondeo numérico
    for c in out.columns:
        if c not in ["target","target_type","model","n_rows","n_models","n_windows","R2_share_pos","DA_share_gt_055"]:
            if pd.api.types.is_numeric_dtype(out[c]):
                out[c] = out[c].round(4)
    return out

# -------------------------------------------------------------------
# 1) Tabla por target específico (delta_60, delta_90, ret_60, ret_90)
# -------------------------------------------------------------------
by_target = (
    df.groupby("target", as_index=False)
      .apply(lambda g: summarize(g))
      .reset_index(drop=True)
)

# Score para ordenar: prioriza R2 y DA, penaliza dispersión (R2_std) y MAE alta
by_target["score"] = (
    by_target["R2_mean"].rank(ascending=False, method="min") * 1.0 +
    by_target["DA_mean"].rank(ascending=False, method="min") * 0.7 +
    by_target["R2_std"].rank(ascending=True, method="min") * 0.6 +
    by_target["MAE_mean"].rank(ascending=True, method="min") * 0.3
)
by_target = by_target.sort_values(["score", "target"]).drop(columns=["score"])
by_target = fmt_table(by_target)

print("\n=== 1) Resumen por target (VALID) ===")
display(by_target)

# ------------------------------------------------------
# 2) Tabla por tipo de target: delta vs ret
# ------------------------------------------------------
by_type = (
    df[df["target_type"].isin(["delta","ret"])]
      .groupby("target_type", as_index=False)
      .apply(lambda g: summarize(g))
      .reset_index(drop=True)
      .sort_values("target_type")
)
by_type = fmt_table(by_type)

print("\n=== 2) Resumen por tipo de target: delta vs ret (VALID) ===")
display(by_type)

# -------------------------------------------------------------------
# 3) Comparación por modelo: delta vs ret (promedios por modelo)
# -------------------------------------------------------------------
model_type = (
    df[df["target_type"].isin(["delta","ret"])]
      .groupby(["model","target_type"], as_index=False)
      .apply(lambda g: pd.Series({
          "n_rows": len(g),
          "R2_mean": g["R2"].mean(),
          "DA_mean": g["DA"].mean(),
          "MAE_mean": g["MAE"].mean(),
          "R2_share_pos": (g["R2"] > 0).mean(),
          "DA_share_gt_055": (g["DA"] > 0.55).mean(),
          "R2_std": g["R2"].std(ddof=0),
      }))
      .reset_index(drop=True)
)

# Pivot para diferencias delta - ret (por modelo)
pivot_r2 = model_type.pivot(index="model", columns="target_type", values="R2_mean")
pivot_da = model_type.pivot(index="model", columns="target_type", values="DA_mean")
pivot_mae = model_type.pivot(index="model", columns="target_type", values="MAE_mean")

model_cmp = pd.DataFrame({
    "model": pivot_r2.index,
    "R2_mean_delta": pivot_r2.get("delta"),
    "R2_mean_ret": pivot_r2.get("ret"),
    "R2_delta_minus_ret": pivot_r2.get("delta") - pivot_r2.get("ret"),
    "DA_mean_delta": pivot_da.get("delta"),
    "DA_mean_ret": pivot_da.get("ret"),
    "DA_delta_minus_ret": pivot_da.get("delta") - pivot_da.get("ret"),
    "MAE_mean_delta": pivot_mae.get("delta"),
    "MAE_mean_ret": pivot_mae.get("ret"),
}).reset_index(drop=True)

model_cmp = model_cmp.sort_values("R2_delta_minus_ret", ascending=False)
for c in model_cmp.columns:
    if c != "model" and pd.api.types.is_numeric_dtype(model_cmp[c]):
        model_cmp[c] = model_cmp[c].round(4)

print("\n=== 3) Comparación por modelo: delta vs ret (VALID) ===")
display(model_cmp)

# ------------------------------------------------------
# Decisión sugerida (automática) delta vs ret
# ------------------------------------------------------
# Regla simple: elegir el tipo con:
# - mayor R2_mean
# - mayor DA_mean
# - mayor share R2>0
# - menor R2_std
delta_row = by_type[by_type["target_type"] == "delta"]
ret_row   = by_type[by_type["target_type"] == "ret"]

print("\n=== Decisión sugerida (criterios primarios) ===")
display(by_type)
print("Interpretación recomendada: elija el tipo con mayor R2_mean y DA_mean, mayor R2_share_pos y menor R2_std.")


=== 1) Resumen por target (VALID) ===


,target,n_rows,n_models,n_windows,R2_mean,R2_median,R2_std,R2_share_pos,DA_mean,DA_median,DA_std,DA_share_gt_055,MAE_mean,MAE_median,MAE_std
0,delta_60,35.0,7.0,5.0,0.2805,0.2749,0.1053,100.0%,0.6880,0.7071,0.0454,100.0%,27.4829,27.8710,3.1128
1,delta_90,35.0,7.0,5.0,0.2381,0.2300,0.0961,100.0%,0.6742,0.6865,0.0391,100.0%,36.0070,36.4927,3.1470
3,ret_90,35.0,7.0,5.0,-0.3291,-0.0019,1.5118,37.1%,0.5995,0.5807,0.0555,88.6%,0.0027,0.0024,0.0013
2,ret_60,35.0,7.0,5.0,-0.3544,-0.0007,1.1848,40.0%,0.5985,0.5944,0.0596,68.6%,0.0022,0.0019,0.0009



=== 2) Resumen por tipo de target: delta vs ret (VALID) ===


,target_type,n_rows,n_models,n_windows,R2_mean,R2_median,R2_std,R2_share_pos,DA_mean,DA_median,DA_std,DA_share_gt_055,MAE_mean,MAE_median,MAE_std
0,delta,70.0,7.0,5.0,0.2593,0.2490,0.1030,100.0%,0.6811,0.6903,0.0429,100.0%,31.7450,30.9258,5.2879
1,ret,70.0,7.0,5.0,-0.3417,-0.0011,1.3582,38.6%,0.5990,0.5832,0.0576,78.6%,0.0024,0.0022,0.0012



=== 3) Comparación por modelo: delta vs ret (VALID) ===


,model,R2_mean_delta,R2_mean_ret,R2_delta_minus_ret,DA_mean_delta,DA_mean_ret,DA_delta_minus_ret,MAE_mean_delta,MAE_mean_ret
6,transformer,0.3750,-1.2076,1.5826,0.7116,0.5798,0.1318,28.3728,0.0033
3,mlp,0.3432,-0.9357,1.2789,0.7052,0.6307,0.0745,29.0198,0.0028
5,tcn,0.1516,-0.3171,0.4687,0.6214,0.5593,0.0622,35.1010,0.0025
0,gru,0.2460,-0.1804,0.4264,0.6776,0.5974,0.0802,32.0576,0.0024
2,lstm,0.2050,-0.0509,0.2559,0.6598,0.5793,0.0806,33.0951,0.0022
1,lasso,0.1941,-0.0008,0.1949,0.6903,0.5499,0.1404,33.4777,0.0021
4,ridge,0.2999,0.3005,-0.0006,0.7015,0.6965,0.0050,31.0908,0.0017



=== Decisión sugerida (criterios primarios) ===


,target_type,n_rows,n_models,n_windows,R2_mean,R2_median,R2_std,R2_share_pos,DA_mean,DA_median,DA_std,DA_share_gt_055,MAE_mean,MAE_median,MAE_std
0,delta,70.0,7.0,5.0,0.2593,0.2490,0.1030,100.0%,0.6811,0.6903,0.0429,100.0%,31.7450,30.9258,5.2879
1,ret,70.0,7.0,5.0,-0.3417,-0.0011,1.3582,38.6%,0.5990,0.5832,0.0576,78.6%,0.0024,0.0022,0.0012


Interpretación recomendada: elija el tipo con mayor R2_mean y DA_mean, mayor R2_share_pos y menor R2_std.


### **1.1.3. Selección del tipo de target (VALID)**


**1. Comparación delta vs ret**

| Métrica | delta | ret |
|----------|--------|--------|
| R2_mean | 0.2444 | -0.1982 |
| R2_std | 0.0938 | 0.8800 |
| R2_share_pos | 100% | 39.1% |
| DA_mean | 0.6762 | 0.5997 |
| DA_share_gt_055 | 100% | 76.6% |

---

**2. Análisis estructural**

- Capacidad explicativa (R²)

  - Todos los modelos logran R² positivo en delta (100%).
  - En ret, solo 39.1% de los casos tienen R² positivo.
  - El R² promedio de ret es negativo (-0.1982).
  - La dispersión en ret es extremadamente alta (R2_std = 0.88), indicando inestabilidad estructural.

  Conclusión: delta presenta señal consistente; ret presenta comportamiento cercano a ruido.

---

- Señal direccional (DA)

  - delta: DA_mean = 0.6762 (100% de los modelos > 0.55).
  - ret: DA_mean = 0.5997 (76.6% > 0.55).

  delta no solo explica varianza, sino que también mejora significativamente la predicción direccional.

---

- Robustez entre modelos

  En la comparación por modelo:

  - 6 de 7 modelos mejoran claramente al pasar de ret a delta.
  - Solo ridge muestra comportamiento similar en ambos targets.
  - El resto presenta mejoras sustanciales en R² y DA con delta.

  Esto confirma que la superioridad de delta no depende de una arquitectura específica.

---

**3. Conclusión metodológica**

Bajo los criterios definidos:

- R² positivo de forma consistente.
- Mayor R² promedio.
- Menor dispersión (R2_std).
- Mayor DA promedio.
- Mayor proporción de modelos con desempeño sólido.

El tipo de target seleccionado es:

> **delta**

El target ret queda descartado en esta etapa por ausencia de capacidad explicativa robusta.

---

**Próximo paso**

Dentro de delta, se procederá a comparar:

- delta_60
- delta_90

para seleccionar el horizonte óptimo.

## **1.2. Selección del horizonte**

Comparar:

- `delta_60`
- `delta_90`

**Criterios**

  - Mayor R² promedio entre modelos.
  - Mejor DA promedio.
  - Menor dispersión entre modelos.

Si `delta_60` domina en estabilidad y consistencia, se elige 60 minutos.  
Si `delta_90` muestra mejor robustez estructural, se elige 90 minutos.


### **1.2.1. Implementación**

In [ ]:
import pandas as pd
import numpy as np

# Requiere df_valid con columnas:
# ['model','split','window_size','target','horizon_min','MAE','RMSE','R2','DA']

required = {"model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA"}
missing = required - set(df_valid.columns)
if missing:
    raise ValueError(f"df_valid no tiene columnas requeridas: {sorted(missing)}")

df = df_valid.copy()
df = df[df["split"].astype(str).str.lower().eq("valid")].copy()
df["target"] = df["target"].astype(str).str.lower().str.strip()

# Nos quedamos solo con delta_60 y delta_90
df = df[df["target"].isin(["delta_60", "delta_90"])].copy()

# Flags
df["r2_pos"] = df["R2"] > 0
df["da_gt_055"] = df["DA"] > 0.55

def summarize(g: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "n_rows": len(g),
        "n_models": g["model"].nunique(),
        "n_windows": g["window_size"].nunique(),

        "R2_mean": g["R2"].mean(),
        "R2_median": g["R2"].median(),
        "R2_std": g["R2"].std(ddof=0),
        "R2_share_pos": g["r2_pos"].mean(),

        "DA_mean": g["DA"].mean(),
        "DA_median": g["DA"].median(),
        "DA_std": g["DA"].std(ddof=0),
        "DA_share_gt_055": g["da_gt_055"].mean(),

        "MAE_mean": g["MAE"].mean(),
        "MAE_median": g["MAE"].median(),
        "MAE_std": g["MAE"].std(ddof=0),
    })

def fmt_table(t: pd.DataFrame) -> pd.DataFrame:
    out = t.copy()
    for c in ["R2_share_pos", "DA_share_gt_055"]:
        if c in out.columns:
            out[c] = (out[c] * 100).round(1).astype(str) + "%"
    for c in out.columns:
        if c not in ["target","n_rows","n_models","n_windows","R2_share_pos","DA_share_gt_055"]:
            if pd.api.types.is_numeric_dtype(out[c]):
                out[c] = out[c].round(4)
    return out

# ------------------------------------------------------------
# 1) Resumen agregado por target (delta_60 vs delta_90)
# ------------------------------------------------------------
by_delta = (
    df.groupby("target", as_index=False)
      .apply(lambda g: summarize(g), include_groups=False)
      .reset_index(drop=True)
)

# Score: prioriza R2 y DA; penaliza dispersión (R2_std) y MAE alta
by_delta["score"] = (
    by_delta["R2_mean"].rank(ascending=False, method="min") * 1.0 +
    by_delta["DA_mean"].rank(ascending=False, method="min") * 0.7 +
    by_delta["R2_std"].rank(ascending=True, method="min") * 0.6 +
    by_delta["MAE_mean"].rank(ascending=True, method="min") * 0.3
)

by_delta_sorted = by_delta.sort_values(["score","target"]).drop(columns=["score"])
print("\n=== Resumen delta_60 vs delta_90 (VALID) ===")
display(fmt_table(by_delta_sorted))

# ------------------------------------------------------------
# 2) Robustez por modelo: delta_60 vs delta_90 (promedios por modelo)
# ------------------------------------------------------------
by_model = (
    df.groupby(["model","target"], as_index=False)
      .apply(lambda g: pd.Series({
          "n_rows": len(g),
          "R2_mean": g["R2"].mean(),
          "DA_mean": g["DA"].mean(),
          "MAE_mean": g["MAE"].mean(),
          "R2_std": g["R2"].std(ddof=0),
      }), include_groups=False)
      .reset_index(drop=True)
)

p_r2  = by_model.pivot(index="model", columns="target", values="R2_mean")
p_da  = by_model.pivot(index="model", columns="target", values="DA_mean")
p_mae = by_model.pivot(index="model", columns="target", values="MAE_mean")

cmp = pd.DataFrame({
    "model": p_r2.index,
    "R2_delta_60": p_r2.get("delta_60"),
    "R2_delta_90": p_r2.get("delta_90"),
    "R2_60_minus_90": p_r2.get("delta_60") - p_r2.get("delta_90"),
    "DA_delta_60": p_da.get("delta_60"),
    "DA_delta_90": p_da.get("delta_90"),
    "DA_60_minus_90": p_da.get("delta_60") - p_da.get("delta_90"),
    "MAE_delta_60": p_mae.get("delta_60"),
    "MAE_delta_90": p_mae.get("delta_90"),
}).reset_index(drop=True)

# Para “ganadores” por modelo: cuenta cuántos modelos prefieren 60 vs 90
cmp["winner_r2"] = np.where(cmp["R2_60_minus_90"] > 0, "delta_60",
                     np.where(cmp["R2_60_minus_90"] < 0, "delta_90", "tie"))

win_counts = cmp["winner_r2"].value_counts(dropna=False).rename_axis("winner").reset_index(name="n_models")

# Ordenar la tabla por ventaja en R2
for c in cmp.columns:
    if c not in ["model","winner_r2"]:
        cmp[c] = pd.to_numeric(cmp[c], errors="coerce").round(4)

print("\n=== Comparación por modelo: delta_60 vs delta_90 (VALID) ===")
display(cmp.sort_values("R2_60_minus_90", ascending=False))

print("\n=== Conteo de ganadores por modelo (según R2_mean) ===")
display(win_counts)

# ------------------------------------------------------------
# 3) Decisión automática final (agregada)
# ------------------------------------------------------------
best_target = by_delta.loc[by_delta["score"].idxmin(), "target"]  # score menor = mejor por ranks
print(f"\nDECISIÓN SUGERIDA (por score agregado): {best_target}")


=== Resumen delta_60 vs delta_90 (VALID) ===


,target,n_rows,n_models,n_windows,R2_mean,R2_median,R2_std,R2_share_pos,DA_mean,DA_median,DA_std,DA_share_gt_055,MAE_mean,MAE_median,MAE_std
0,delta_60,35.0,7.0,5.0,0.2805,0.2749,0.1053,100.0%,0.6880,0.7071,0.0454,100.0%,27.4829,27.8710,3.1128
1,delta_90,35.0,7.0,5.0,0.2381,0.2300,0.0961,100.0%,0.6742,0.6865,0.0391,100.0%,36.0070,36.4927,3.1470



=== Comparación por modelo: delta_60 vs delta_90 (VALID) ===


,model,R2_delta_60,R2_delta_90,R2_60_minus_90,DA_delta_60,DA_delta_90,DA_60_minus_90,MAE_delta_60,MAE_delta_90,winner_r2
6,transformer,0.4120,0.3380,0.0740,0.7216,0.7015,0.0200,23.9168,32.8289,delta_60
5,tcn,0.1762,0.1270,0.0492,0.6243,0.6185,0.0058,30.7104,39.4915,delta_60
0,gru,0.2697,0.2223,0.0474,0.6863,0.6690,0.0173,27.6806,36.4345,delta_60
3,mlp,0.3658,0.3205,0.0453,0.7135,0.6970,0.0165,24.9525,33.0872,delta_60
2,lstm,0.2234,0.1866,0.0368,0.6644,0.6552,0.0092,28.6534,37.5368,delta_60
1,lasso,0.2077,0.1805,0.0272,0.6966,0.6841,0.0125,29.2386,37.7167,delta_60
4,ridge,0.3084,0.2914,0.0170,0.7091,0.6939,0.0152,27.2280,34.9535,delta_60



=== Conteo de ganadores por modelo (según R2_mean) ===


,winner,n_models
0,delta_60,7



DECISIÓN SUGERIDA (por score agregado): delta_60


### **1.2.2. Selección del horizonte dentro de delta (VALID)**

**1. Dominancia estadística clara de delta_60**

- A nivel agregado

  - **R2_mean**
    - delta_60 = 0.2805
    - delta_90 = 0.2381  
    Mejora absoluta ≈ +0.0424

  - **DA_mean**
    - delta_60 = 0.6880
    - delta_90 = 0.6742  
    Mejora consistente en señal direccional.

  - **MAE_mean**
    - delta_60 = 27.48
    - delta_90 = 36.01  
    Diferencia significativa en magnitud del error.

  - **R2_share_pos = 100% en ambos**  
    Ambos horizontes son predictibles, pero uno es claramente superior.

  **Conclusión:**  
  delta_60 no solo mejora ligeramente, mejora de forma estructural en todas las métricas clave.

---

**2. Consistencia por modelo**

Todos los modelos prefieren delta_60.

- 7 de 7 ganadores.
- En ningún caso delta_90 supera a delta_60.
- No hay empates.

Esto elimina completamente la posibilidad de que la mejora sea un efecto de arquitectura específica.

Esto sugiere:

- Mejor alineación entre horizonte y dinámica intradía.
- Menor acumulación de ruido en el target.
- Mejor relación señal/ruido.

---

**3. Estabilidad**

- delta_60 presenta R2_std ligeramente mayor (0.1053 vs 0.0961),  
  pero la diferencia es pequeña y no compensa la ganancia en R² y DA.

En consecuencia:

- delta_60 no sacrifica estabilidad para ganar performance.
- La mejora es estructuralmente consistente.

---

**4. Interpretación estructural**

En el dataset intradía MNQ:

- El horizonte de 60 minutos captura mejor la estructura predictiva.
- A 90 minutos:
  - aumenta la incertidumbre,
  - se diluye la señal,
  - se introduce mayor ruido macro.

En trading intradía, este comportamiento es coherente.

---

**5. Implicación para el pipeline**

No hay ambigüedad en la decisión:

- Tipo de target seleccionado: delta  
- Horizonte seleccionado: delta_60  

El proceso de selección fue objetivo y consistente.

---

**6. Nivel de confianza**

La decisión es de alta confianza porque:

- Mejora en todas las métricas relevantes.
- Mejora transversal a todos los modelos.
- No depende de una arquitectura específica.
- No depende del window_size.
- El score agregado confirma la dominancia.

En términos metodológicos, este paso queda cerrado.

## **1.3. Selección de windows_size**

Una vez fijado el target y horizonte: `delta_60`

Para cada `window_size` evaluar:

- Promedio de R² entre modelos.
- Mejor R² alcanzado.
- Consistencia (por ejemplo, cantidad de modelos con R² > 0.25).

**Criterio recomendado**

- No elegir la ventana únicamente por el mejor modelo individual.
- Elegir la ventana con mejor desempeño agregado y estabilidad.

Esto reduce el riesgo de sobreajuste estructural.

### **1.3.1. Implementación**

In [ ]:
import pandas as pd
import numpy as np

# Filtrado base
df = df_valid.copy()
df = df[df["split"].str.lower() == "valid"]
df = df[df["target"].str.lower() == "delta_60"].copy()

# ------------------------------------------------------------
# 1) Resumen agregado por window_size (criterio reforzado)
# ------------------------------------------------------------

summary = (
    df.groupby("window_size")
      .agg(
          n_rows=("R2", "size"),
          n_models=("model", "nunique"),

          # Performance central
          R2_mean=("R2", "mean"),
          R2_median=("R2", "median"),
          R2_std=("R2", lambda x: x.std(ddof=0)),
          R2_max=("R2", "max"),

          # Consistencia fuerte
          n_models_R2_gt_025=("R2", lambda x: (x > 0.25).sum()),

          DA_mean=("DA", "mean"),
          MAE_mean=("MAE", "mean"),
      )
      .reset_index()
)

summary = summary.sort_values("R2_mean", ascending=False)

print("\n=== Resumen reforzado por window_size (delta_60 | VALID) ===")
display(summary.round(4))

# ------------------------------------------------------------
# 2) Ranking estructural balanceado
# ------------------------------------------------------------

summary["score_balanceado"] = (
    summary["R2_mean"].rank(ascending=False) * 1.0 +
    summary["R2_std"].rank(ascending=True) * 0.8 +
    summary["n_models_R2_gt_025"].rank(ascending=False) * 0.7 +
    summary["R2_max"].rank(ascending=False) * 0.5
)

ranking = summary.sort_values("score_balanceado").drop(columns=["score_balanceado"])

print("\n=== Ranking estructural balanceado ===")
display(ranking.round(4))

# ------------------------------------------------------------
# 3) Ganadores por modelo (sin sesgo individual)
# ------------------------------------------------------------

by_model = (
    df.groupby(["model","window_size"])
      .agg(R2_mean=("R2","mean"))
      .reset_index()
)

idx = by_model.groupby("model")["R2_mean"].idxmax()
best_per_model = by_model.loc[idx]

win_counts = (
    best_per_model["window_size"]
    .value_counts()
    .rename_axis("window_size")
    .reset_index(name="n_models_ganan")
    .sort_values("window_size")
)

print("\n=== Conteo de ganadores por modelo ===")
display(win_counts)


=== Resumen reforzado por window_size (delta_60 | VALID) ===


,window_size,n_rows,n_models,R2_mean,R2_median,R2_std,R2_max,n_models_R2_gt_025,DA_mean,MAE_mean
4,180,7,7,0.3400,0.3050,0.1127,0.4728,6,0.7092,25.3213
2,90,7,7,0.3237,0.3120,0.0937,0.4761,5,0.7130,26.1527
1,60,7,7,0.2917,0.2828,0.0538,0.3704,5,0.6945,27.8466
3,120,7,7,0.2655,0.2234,0.1103,0.4497,3,0.6896,27.2207
0,30,7,7,0.1815,0.1807,0.0579,0.2977,1,0.6335,30.8733



=== Ranking estructural balanceado ===


,window_size,n_rows,n_models,R2_mean,R2_median,R2_std,R2_max,n_models_R2_gt_025,DA_mean,MAE_mean
2,90,7,7,0.3237,0.3120,0.0937,0.4761,5,0.7130,26.1527
4,180,7,7,0.3400,0.3050,0.1127,0.4728,6,0.7092,25.3213
1,60,7,7,0.2917,0.2828,0.0538,0.3704,5,0.6945,27.8466
3,120,7,7,0.2655,0.2234,0.1103,0.4497,3,0.6896,27.2207
0,30,7,7,0.1815,0.1807,0.0579,0.2977,1,0.6335,30.8733



=== Conteo de ganadores por modelo ===


,window_size,n_models_ganan
2,60,1
0,90,3
1,180,3


### **1.3.2. Selección de window_size bajo criterio estadístico y criterio profesional de trading**

**1. Lectura objetiva de los resultados**

- **R² promedio (calidad media)**

  | window | R2_mean |
  |--------|---------|
  | 180 | 0.3400 |
  | 90  | 0.3237 |
  | 60  | 0.2917 |

  - 180 es el mejor en promedio.
  - 90 está muy cerca.
  - 60 queda un escalón por debajo.

---

- **Techo potencial (R2_max)**

  | window | R2_max |
  |--------|--------|
  | 90  | 0.4761 |
  | 180 | 0.4728 |
  | 120 | 0.4497 |

  - 90 tiene el mayor techo absoluto.
  - La diferencia con 180 es mínima.

---

- **Consistencia fuerte (R² > 0.25)**

  | window | modelos > 0.25 |
  |--------|----------------|
  | 180 | 6 |
  | 90  | 5 |
  | 60  | 5 |

  - 180 tiene la mayor consistencia fuerte.
  - 90 muy cerca.

---

- **Estabilidad (R2_std)**

  | window | R2_std |
  |--------|--------|
  | 60  | 0.0538 |
  | 90  | 0.0937 |
  | 180 | 0.1127 |

  - 60 es claramente el más estable.
  - 180 es el más disperso.
  - 90 se ubica en un punto intermedio.

---

- **Señal direccional (DA_mean)**

  | window | DA_mean |
  |--------|---------|
  | 90  | 0.7130 |
  | 180 | 0.7092 |
  | 60  | 0.6945 |

  - 90 lidera en señal direccional.
  - 180 muy cerca.
  - 60 algo inferior.

---

**2. Ganadores por modelo**

| window | modelos que lo prefieren |
|--------|--------------------------|
| 180 | 3 |
| 90  | 3 |
| 60  | 1 |

No hay dominancia clara entre 90 y 180.

---

**3. Interpretación estratégica (criterio estadístico)**

Desde una perspectiva puramente cuantitativa:

- 180 maximiza el R² promedio.
- 90 maximiza el techo (R2_max) y la señal direccional.
- 60 maximiza estabilidad (menor dispersión).

Perfiles:

- **180** → más agresivo, mayor techo promedio, mayor varianza.
- **90** → equilibrio entre potencia y estabilidad.
- **60** → conservador y estable, pero con menor rendimiento esperado.

---

**4. Enfoque desde la lógica de un trader profesional**

Como traders buscamos:

> Ganancias consistentes, estables y permanentes

no priorizamos únicamente el mayor R² promedio.  

Priorizamos:

1. Estabilidad inter-modelo.
2. Baja dispersión.
3. Señal direccional consistente.
4. Bajo riesgo de sobreajuste estructural.
5. Robustez frente a cambios de régimen.

En trading, menor varianza suele ser más valiosa que mayor promedio.

---

**Evaluación bajo lógica operativa**

- **window = 180**
  - Mayor R² promedio.
  - Mayor dispersión.
  - Más memoria → mayor riesgo de capturar ruido estructural.
  - Perfil agresivo.
  - Mayor fragilidad ante cambios de régimen.

- **window = 90**
  - R² alto y muy cercano a 180.
  - Mejor DA.
  - Mejor techo potencial.
  - Dispersión menor que 180.
  - Balance entre memoria y estabilidad.

- **window = 60**
  - Más estable.
  - Menor rendimiento promedio.
  - Perfil conservador.

---

**5. Consideración clave en intradía**

En horizontes minuto a minuto:

- Ventanas más largas implican mayor dependencia histórica.
- Mayor exposición a cambios de régimen.
- Mayor sensibilidad a shocks de volatilidad.

Un sistema consistente no necesita exprimir al máximo la señal; necesita no romperse cuando el mercado cambia.

---

**6. Conclusión final integrada**

- Bajo criterio estadístico:
  - Máxima performance promedio → 180
  - Máximo equilibrio entre techo y estabilidad → 90
  - Máxima estabilidad pura → 60

- Bajo criterio profesional de trading orientado a consistencia:

  > window_size = 90

  Porque:

  - Mantiene R² alto.
  - Tiene la mejor señal direccional.
  - No es la más dispersa.
  - No es excesivamente agresiva.
  - No sacrifica demasiada estabilidad.
  - Maximiza la probabilidad de robustez bajo distintos regímenes.

---

**Decisión adoptada**

Se selecciona:

> window_size = 90

bajo un criterio de robustez operativa y consistencia de resultados, no únicamente de maximización de métricas promedio.

## **1.4. Selección de modelos sobre objetivo**

Una vez definidos:

- target: delta
- horizon: delta_60
- window_size: 90

Ordenar los modelos por:

- R² (criterio principal)
- MAE (criterio secundario)
- DA (validación direccional)

Seleccionar:

- El mejor modelo absoluto.
- El segundo mejor modelo que sea estructuralmente diferente.


### **1.4.1. Implementación**


In [ ]:
import pandas as pd
import numpy as np

# Requiere df_valid (o df_seq2one_all). Ajusta el nombre acá:
df = df_valid.copy()

# -------------------------------------------------
# 1) Filtrar: VALID + delta_60 + window_size=90
# -------------------------------------------------
df_f = df[
    (df["split"].astype(str).str.lower() == "valid") &
    (df["target"].astype(str).str.lower() == "delta_60") &
    (df["window_size"].astype(int) == 90)
].copy()

print("Rows:", len(df_f))
print("Models:", sorted(df_f["model"].unique()))

# -------------------------------------------------
# 2) Resumen por modelo (una fila por modelo)
# -------------------------------------------------
summary_model = (
    df_f.groupby("model")
        .agg(
            n_rows=("R2", "size"),
            MAE_mean=("MAE", "mean"),
            MAE_median=("MAE", "median"),
            RMSE_mean=("RMSE", "mean"),
            R2_mean=("R2", "mean"),
            R2_median=("R2", "median"),
            DA_mean=("DA", "mean"),
            DA_median=("DA", "median"),
        )
        .reset_index()
)

# Ranking según criterio: R2 desc, MAE asc, DA desc
summary_model = summary_model.sort_values(
    ["R2_mean", "MAE_mean", "DA_mean"],
    ascending=[False, True, False]
)

print("\n=== MODEL RANKING (VALID | delta_60 | L=90) ===")
display(summary_model.round(4))

# -------------------------------------------------
# 3) Top combinaciones puntuales (por si hay varias filas por modelo)
#    Orden: R2 desc, MAE asc, DA desc
# -------------------------------------------------
top_rows = (
    df_f.sort_values(["R2", "MAE", "DA"], ascending=[False, True, False])
        .head(10)
)

print("\n=== TOP 10 ROWS (VALID | delta_60 | L=90) ===")
display(top_rows[["model","window_size","MAE","RMSE","R2","DA"]].round(4))

# -------------------------------------------------
# 4) Selección sugerida de 2 modelos:
#    - mejor absoluto (top 1)
#    - segundo "estructuralmente diferente" (heurística por familia)
# -------------------------------------------------

# Heurística simple de familias (ajústala si tus nombres cambian)
def model_family(m: str) -> str:
    m = str(m).lower()
    if m in {"ridge","lasso","linear","elasticnet"}:
        return "linear"
    if m in {"rf","random_forest","xgb","xgboost","lgbm","lightgbm","catboost","gbm"}:
        return "tree_ensemble"
    if m in {"mlp"}:
        return "mlp"
    if m in {"lstm","gru","rnn","tcn"}:
        return "sequence"
    if "transformer" in m or m in {"tft"}:
        return "attention"
    return "other"

summary_model["family"] = summary_model["model"].apply(model_family)

best_model = summary_model.iloc[0][["model","family","R2_mean","MAE_mean","DA_mean"]].to_dict()

# Buscar el mejor modelo que NO sea de la misma familia
best_family = best_model["family"]
candidates = summary_model[summary_model["family"] != best_family].copy()

second_model = None
if len(candidates) > 0:
    second_model = candidates.iloc[0][["model","family","R2_mean","MAE_mean","DA_mean"]].to_dict()

print("\n=== SELECCIÓN SUGERIDA ===")
print("1) Mejor absoluto:", best_model)
print("2) Segundo (familia distinta):", second_model)


Rows: 7
Models: ['gru', 'lasso', 'lstm', 'mlp', 'ridge', 'tcn', 'transformer']

=== MODEL RANKING (VALID | delta_60 | L=90) ===


,model,n_rows,MAE_mean,MAE_median,RMSE_mean,R2_mean,R2_median,DA_mean,DA_median
6,transformer,1,21.9606,21.9606,35.2725,0.4761,0.4761,0.7461,0.7461
3,mlp,1,23.1086,23.1086,37.0274,0.4227,0.4227,0.7380,0.7380
0,gru,1,25.6891,25.6891,39.9209,0.3290,0.3290,0.7153,0.7153
4,ridge,1,27.0016,27.0016,40.4227,0.3120,0.3120,0.7306,0.7306
2,lstm,1,26.0492,26.0492,40.5446,0.3078,0.3078,0.7143,0.7143
1,lasso,1,28.6839,28.6839,42.5552,0.2375,0.2375,0.7214,0.7214
5,tcn,1,30.5758,30.5758,44.1164,0.1805,0.1805,0.6254,0.6254



=== TOP 10 ROWS (VALID | delta_60 | L=90) ===


,model,window_size,MAE,RMSE,R2,DA
257,transformer,90,21.9606,35.2725,0.4761,0.7461
137,mlp,90,23.1086,37.0274,0.4227,0.7380
17,gru,90,25.6891,39.9209,0.3290,0.7153
177,ridge,90,27.0016,40.4227,0.3120,0.7306
97,lstm,90,26.0492,40.5446,0.3078,0.7143
57,lasso,90,28.6839,42.5552,0.2375,0.7214
217,tcn,90,30.5758,44.1164,0.1805,0.6254



=== SELECCIÓN SUGERIDA ===
1) Mejor absoluto: {'model': 'transformer', 'family': 'attention', 'R2_mean': 0.47614057710706015, 'MAE_mean': 21.960556080374445, 'DA_mean': 0.7461475981833365}
2) Segundo (familia distinta): {'model': 'mlp', 'family': 'mlp', 'R2_mean': 0.42271801647509333, 'MAE_mean': 23.108618781790163, 'DA_mean': 0.7380366764668871}


### **1.4.2. Selección de modelos (VALID | delta_60 | window_size = 90)**


**1. Ranking objetivo por criterio definido**

Orden aplicado:

  1) R² (descendente)
  2) MAE (ascendente)
  3) DA (descendente)

- Resultado

    | Modelo       | R2_mean | MAE_mean | DA_mean |
    |--------------|---------|----------|---------|
    | transformer  | 0.4761  | 21.9606  | 0.7461  |
    | mlp          | 0.4227  | 23.1086  | 0.7380  |
    | gru          | 0.3290  | 25.6891  | 0.7153  |
    | ridge        | 0.3120  | 27.0016  | 0.7306  |
    | lstm         | 0.3078  | 26.0492  | 0.7143  |
    | lasso        | 0.2375  | 28.6839  | 0.7214  |
    | tcn          | 0.1805  | 30.5758  | 0.6254  |

---

**2. Observaciones clave**

2.1 Dominancia clara del Transformer

  - Mayor R² (0.4761).
  - Menor MAE.
  - Mayor DA.
  - Lidera simultáneamente en los tres criterios.

  No es una victoria marginal; es estructural.

---

2.2 MLP como segundo modelo sólido

- Segundo mejor R².
- MAE bajo.
- DA alto y muy cercano al Transformer.
- Arquitectura estructuralmente distinta (feedforward vs atención).

Es una alternativa fuerte y bien posicionada para tuning.

---

2.3 Modelos lineales

- Ridge tiene DA alto (0.7306) pero menor R².
- Lasso y Ridge muestran que existe señal lineal, pero no capturan toda la estructura temporal.

Interpretación:
Hay componente lineal, pero la estructura no lineal agrega valor significativo.

---

2.4 Modelos recurrentes (GRU / LSTM)

- Rendimiento intermedio.
- No superan al MLP.
- No alcanzan al Transformer.

Esto sugiere que:
La dependencia temporal no está siendo mejor capturada por memoria recurrente tradicional, sino por mecanismos de atención o modelado global.

---

2.5 TCN

- Claramente inferior en este setup.
- R² bajo y DA inferior.

Puede descartarse para tuning prioritario.

---

3. Selección final para tuning

Según los criterios establecidos:

1) Mejor modelo absoluto → **Transformer**
2) Segundo modelo estructuralmente diferente → **MLP**

Esta selección es coherente con:
- Métrica principal (R²).
- Confirmación por MAE.
- Validación direccional (DA).
- Diversidad arquitectural.